In [18]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from skimage import io, exposure, color, filters
from skimage.color import rgb2gray
from skimage.filters import sobel, laplace, gaussian

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.models as models
import torchvision.transforms as T
from torchvision import transforms
from torchvision.transforms import Normalize


In [19]:

BASE_DIR = Path(r"C:\Users\oliwa\Documents\capstone\images_for_modeling\combined_deforestation_drivers")

# check
if not BASE_DIR.exists():
    raise FileNotFoundError(f"Could not find base directory: {BASE_DIR!r}")

### Train test split

In [37]:
# locate the class directories with raw, reason for this is I experimented with various types of preprocessing which had different suffixes
raw_dirs = sorted(d for d in BASE_DIR.iterdir() if d.is_dir() and d.name.endswith("raw"))
class_map = {d.name: idx for idx, d in enumerate(raw_dirs)}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [38]:


# 4) build a dataframe from the folder names
rows = []
for cls_name, label in class_map.items():
    folder = BASE_DIR / cls_name
    for img_path in folder.glob("*.png"):
        rows.append({"filepath": str(img_path), "label": label})
df = pd.DataFrame(rows)

# 6) stratified split to preserve ratios
train_df, val_df = train_test_split(
    df,
    test_size=0.25,
    stratify=df["label"],
    random_state=42
)


# save out CSVs
df.to_csv(BASE_DIR / "labels_basic.csv", index=False)
train_df.to_csv(BASE_DIR / "train_labels_basic.csv", index=False)
val_df.to_csv(BASE_DIR / "val_labels_basic.csv", index=False)
#print(f"\nSaved {total} entries to labels_basic.csv")
print(f"Train samples: {len(train_df)} → train_labels_basic.csv")
print(f"Val   samples: {len(val_df)} → val_labels_basic.csv")

Train samples: 536 → train_labels_basic.csv
Val   samples: 179 → val_labels_basic.csv


In [39]:


# point to the folder where your CSVs live
BASE_DIR   = Path(r"C:\Users\oliwa\Documents\capstone\images_for_modeling\combined_deforestation_drivers")
train_csv  = BASE_DIR / "train_labels_basic.csv"

# load your existing train split
train_df = pd.read_csv(train_csv)

# set up 4‐fold stratified cross‐validation
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

# generate and save folds
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label"])):
    df_train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    df_val_fold   = train_df.iloc[val_idx].reset_index(drop=True)

    df_train_fold.to_csv(BASE_DIR / f"fold{fold}_train.csv", index=False)
    df_val_fold.  to_csv(BASE_DIR / f"fold{fold}_val.csv",   index=False)

    print(f"Fold {fold}: {len(df_train_fold)} train / {len(df_val_fold)} val")

Fold 0: 402 train / 134 val
Fold 1: 402 train / 134 val
Fold 2: 402 train / 134 val
Fold 3: 402 train / 134 val


##### Check distribution for folds

In [23]:
# 1) set up
BASE_DIR = Path(r"C:\Users\oliwa\Documents\capstone\images_for_modeling\combined_deforestation_drivers")
n_folds = 4

def print_distribution(df: pd.DataFrame, name: str):
    counts = df["label"].value_counts().sort_index()
    total = counts.sum()
    print(f"{name} distribution:")
    for label, cnt in counts.items():
        pct = cnt / total * 100
        print(f"  label {label}: {cnt} ({pct:.1f}%)")
    print()

# 2) loop through folds
for fold in range(n_folds):
    train_df = pd.read_csv(BASE_DIR / f"fold{fold}_train.csv")
    val_df   = pd.read_csv(BASE_DIR / f"fold{fold}_val.csv")

    print(f"=== Fold {fold} ===")
    print_distribution(train_df, "Train")
    print_distribution(val_df,   "Validation")

=== Fold 0 ===
Train distribution:
  label 0: 45 (10.5%)
  label 1: 117 (27.3%)
  label 2: 46 (10.7%)
  label 3: 203 (47.3%)
  label 4: 18 (4.2%)

Validation distribution:
  label 0: 16 (11.2%)
  label 1: 38 (26.6%)
  label 2: 16 (11.2%)
  label 3: 67 (46.9%)
  label 4: 6 (4.2%)

=== Fold 1 ===
Train distribution:
  label 0: 46 (10.7%)
  label 1: 116 (27.0%)
  label 2: 46 (10.7%)
  label 3: 203 (47.3%)
  label 4: 18 (4.2%)

Validation distribution:
  label 0: 15 (10.5%)
  label 1: 39 (27.3%)
  label 2: 16 (11.2%)
  label 3: 67 (46.9%)
  label 4: 6 (4.2%)

=== Fold 2 ===
Train distribution:
  label 0: 46 (10.7%)
  label 1: 116 (27.0%)
  label 2: 47 (11.0%)
  label 3: 202 (47.1%)
  label 4: 18 (4.2%)

Validation distribution:
  label 0: 15 (10.5%)
  label 1: 39 (27.3%)
  label 2: 15 (10.5%)
  label 3: 68 (47.6%)
  label 4: 6 (4.2%)

=== Fold 3 ===
Train distribution:
  label 0: 46 (10.7%)
  label 1: 116 (27.0%)
  label 2: 47 (11.0%)
  label 3: 202 (47.1%)
  label 4: 18 (4.2%)

Validation

### Image Transformation Definitions for Input to Model

In [24]:
class CSVImageDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

# R,G,B means and std deviations of the original imagenet dataset, required to normalize how the model expects
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.Resize(256),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean,
                std=imagenet_std)
])

val_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean,
                std=imagenet_std)
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [25]:
# # 6) Training & Validation ---------------------------------------------------------
# num_epochs = 30
# for epoch in range(1, num_epochs+1):
#     # --- training ---
#     model.train()
#     running_loss = 0.0
#     for batch_idx, (imgs, labels) in enumerate(train_loader, 1):
#         imgs, labels = imgs.to(device), labels.to(device)
#         optimizer.zero_grad()
#         logits = model(imgs)
#         loss = criterion(logits, labels)
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#         if batch_idx % 10 == 0:
#             avg = running_loss / 10
#             print(f"Epoch {epoch} | Batch {batch_idx}/{len(train_loader)} | loss: {avg:.4f}")
#             running_loss = 0.0

#     # --- validation ---
#     model.eval()
#     val_loss, correct, total = 0.0, 0, 0
#     with torch.no_grad():
#         for imgs, labels in val_loader:
#             imgs, labels = imgs.to(device), labels.to(device)
#             logits = model(imgs)
#             val_loss += criterion(logits, labels).item() * imgs.size(0)
#             preds = logits.argmax(dim=1)
#             correct += (preds == labels).sum().item()
#             total += labels.size(0)
#     val_loss /= total
#     val_acc   = correct / total
#     print(f"Epoch {epoch} → Val loss: {val_loss:.4f}, Val acc: {val_acc:.4f}")


### Training 

In [29]:
import warnings
## really should fix this but in the interest of time just ignored as it didn't cause any actual issues
warnings.filterwarnings("ignore", message="The parameter 'pretrained' is deprecated")
warnings.filterwarnings("ignore", message="Arguments other than a weight enum")
warnings.filterwarnings("ignore", message="The verbose parameter is deprecated")
warnings.filterwarnings("ignore", message=".*weights_only=False.*")

In [48]:
#workaround for a pesky issue with the class mapping
CLASS_DIRS = sorted(
    d.name
    for d in BASE_DIR.iterdir()
    if d.is_dir() and d.name.endswith("_raw") and d.name != "undisturbed_forest_raw"
)

class_map = {name: idx for idx, name in enumerate(CLASS_DIRS)}
class_map["undisturbed_forest"] = len(class_map)
print("Final class_map:", class_map)

Final class_map: {'maize_plantation_raw': 0, 'selective_logging_raw': 1, 'tropical_tree_plantation_raw': 2, 'wildfire_raw': 3, 'undisturbed_forest': 4}


In [50]:



BASE_DIR       = Path(r"C:\Users\oliwa\Documents\capstone\images_for_modeling\combined_deforestation_drivers")
#hyperparameters
num_epochs     = 50
lr             = 3e-5
es_patience    = 5
sched_patience = 4   # for ReduceLROnPlateau
N_FOLDS        = 4



# ─── overall-best trackers ───────────────────────────────────────────────────────
overall_best_acc  = 0.0
overall_best_wts  = None
overall_best_fold = -1

for fold in range(N_FOLDS):
    print(f"\n=== Fold {fold} ===")

    # load fold CSVs 
    train_csv = BASE_DIR / f"fold{fold}_train.csv"
    val_csv   = BASE_DIR / f"fold{fold}_val.csv"

    df_train = pd.read_csv(train_csv)
    df_val   = pd.read_csv(val_csv)
    print("  Train label counts:\n", df_train.label.value_counts().to_dict())
    print("    Val label counts:\n", df_val.label.value_counts().to_dict())

    # build DataLoaders
    train_ds = CSVImageDataset(train_csv, transform=train_transforms)
    val_ds   = CSVImageDataset(val_csv,   transform=val_transforms)
    train_loader = DataLoader(train_ds,
                              batch_size=32,
                              shuffle=True,
                              num_workers=0,
                              pin_memory=True)
    val_loader   = DataLoader(val_ds,
                              batch_size=32,
                              shuffle=False,
                              num_workers=0,
                              pin_memory=True)

    #  init fresh model, optimizer, scheduler for each fold 
    model = models.mobilenet_v3_large(pretrained=True)
    in_feats = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_feats, len(class_map))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=sched_patience,
        verbose=True
    )

    # ─── per-fold best & early-stop trackers 
    best_acc   = 0.0
    best_wts   = copy.deepcopy(model.state_dict())
    es_counter = 0

    # ─── train & validate
    for epoch in range(1, num_epochs + 1):
        # — training —
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        # — validation —
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total   += labels.size(0)
        val_acc = correct / total
        print(f"Fold {fold} Epoch {epoch} → Val acc: {val_acc:.4f}")

        scheduler.step(val_acc)

        if val_acc > best_acc:
            best_acc   = val_acc
            best_wts   = copy.deepcopy(model.state_dict())
            es_counter = 0
        else:
            es_counter += 1
            if es_counter >= es_patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {es_patience} epochs)")
                break

    # save this fold’s best checkpoint
    ckpt_path = BASE_DIR / f"best_mobilenet_fold{fold}.pth"
    torch.save(best_wts, ckpt_path)
    print(f">>> Fold {fold} best acc: {best_acc:.4f} (saved to {ckpt_path.name})")

    if best_acc > overall_best_acc:
        overall_best_acc  = best_acc
        overall_best_wts  = best_wts
        overall_best_fold = fold

# ─── after all folds, save the single best model across folds ───────────────────
if overall_best_wts is not None:
    cv_ckpt = BASE_DIR / "best_mobilenet_cv.pth"
    torch.save(overall_best_wts, cv_ckpt)
    print(f"\n=== Cross-validated best model ===")
    print(f"Fold {overall_best_fold} achieved highest acc: {overall_best_acc:.4f}")
    print(f"Saved overall best checkpoint to {cv_ckpt.name}")
else:
    print("No model checkpoints were saved during cross-validation.")




=== Fold 0 ===
  Train label counts:
 {4: 250, 1: 116, 2: 46, 0: 45, 3: 18}
    Val label counts:
 {4: 88, 1: 39, 0: 16, 2: 15, 3: 6}
Fold 0 Epoch 1 → Val acc: 0.0488
Fold 0 Epoch 2 → Val acc: 0.0732
Fold 0 Epoch 3 → Val acc: 0.1098
Fold 0 Epoch 4 → Val acc: 0.2622
Fold 0 Epoch 5 → Val acc: 0.4756
Fold 0 Epoch 6 → Val acc: 0.6524
Fold 0 Epoch 7 → Val acc: 0.6829
Fold 0 Epoch 8 → Val acc: 0.7317
Fold 0 Epoch 9 → Val acc: 0.7805
Fold 0 Epoch 10 → Val acc: 0.7927
Fold 0 Epoch 11 → Val acc: 0.8110
Fold 0 Epoch 12 → Val acc: 0.8293
Fold 0 Epoch 13 → Val acc: 0.8354
Fold 0 Epoch 14 → Val acc: 0.8354
Fold 0 Epoch 15 → Val acc: 0.8476
Fold 0 Epoch 16 → Val acc: 0.8537
Fold 0 Epoch 17 → Val acc: 0.8659
Fold 0 Epoch 18 → Val acc: 0.8720
Fold 0 Epoch 19 → Val acc: 0.8780
Fold 0 Epoch 20 → Val acc: 0.8841
Fold 0 Epoch 21 → Val acc: 0.8841
Fold 0 Epoch 22 → Val acc: 0.9085
Fold 0 Epoch 23 → Val acc: 0.9146
Fold 0 Epoch 24 → Val acc: 0.9085
Fold 0 Epoch 25 → Val acc: 0.9207
Fold 0 Epoch 26 → Val ac

### Evaluation

In [49]:


BASE_DIR  = Path(r"C:\Users\oliwa\Documents\capstone\images_for_modeling\combined_deforestation_drivers")

# find all checkpoint files
ckpt_paths = sorted(BASE_DIR.glob("best_mobilenet_fold*.pth"))
if not ckpt_paths:
    raise FileNotFoundError(f"No checkpoints found in {BASE_DIR} matching 'best_mobilenet_fold*.pth'")

all_true = []
all_pred = []
fold_accuracies = []
class_names = [cls for cls, idx in sorted(class_map.items(), key=lambda x: x[1])]

for ckpt in ckpt_paths:
    # extract fold number from filename
    stem = ckpt.stem  # e.g. "best_mobilenet_fold0"
    fold = int(stem.split("fold")[-1])
    print(f"\n=== Fold {fold} ===")

    # load this fold's validation set
    val_csv = BASE_DIR / f"fold{fold}_val.csv"
    if not val_csv.exists():
        raise FileNotFoundError(f"Missing validation CSV for fold {fold}: {val_csv}")
    val_ds = CSVImageDataset(val_csv, transform=val_transforms)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

    # rebuild model architecture and load weights
    model = models.mobilenet_v3_large(pretrained=True)
    in_feats = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_feats, len(class_map))
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model = model.to(device).eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            preds  = logits.argmax(dim=1).cpu().tolist()
            y_pred.extend(preds)
            y_true.extend(labels.tolist())

    acc = accuracy_score(y_true, y_pred)
    print(f"Fold {fold} accuracy: {acc:.4f}")
    fold_accuracies.append(acc)

    all_true.extend(y_true)
    all_pred.extend(y_pred)


overall_acc = accuracy_score(all_true, all_pred)
print(f"\nOverall CV accuracy: {overall_acc:.4f}")


cm = confusion_matrix(all_true, all_pred)
df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
# print("\nConfusion Matrix (rows=true, cols=pred):")
# print(df_cm)

# per-class precision/recall/F1
print("\nClassification Report:")
print(classification_report(all_true, all_pred, target_names=class_names))



=== Fold 0 ===
Fold 0 accuracy: 0.9451

=== Fold 1 ===
Fold 1 accuracy: 0.9286

=== Fold 2 ===
Fold 2 accuracy: 0.9390

=== Fold 3 ===
Fold 3 accuracy: 0.9490

Overall CV accuracy: 0.9405

Classification Report:
                              precision    recall  f1-score   support

        maize_plantation_raw       0.79      0.85      0.82        61
       selective_logging_raw       0.90      0.98      0.94       155
tropical_tree_plantation_raw       0.88      0.70      0.78        61
                wildfire_raw       0.89      0.67      0.76        24
          undisturbed_forest       1.00      1.00      1.00       338

                    accuracy                           0.94       639
                   macro avg       0.89      0.84      0.86       639
                weighted avg       0.94      0.94      0.94       639



#### Extra section out of curiousty for explainability , GPT wrote most of this. Generates some images for interpretation and saves them

In [21]:


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
EXPLAIN_DIR = BASE_DIR / "explanations"
EXPLAIN_DIR.mkdir(exist_ok=True)

# ---- helpers -------------------------------------------------------------------
def denorm(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean = torch.tensor(mean, device=img_tensor.device).view(1,3,1,1)
    std  = torch.tensor(std,  device=img_tensor.device).view(1,3,1,1)
    x = (img_tensor * std + mean).clamp(0,1)
    return (x * 255).byte().permute(0,2,3,1).cpu().numpy()  # (B,H,W,3) uint8

def apply_clahe(rgb_uint8, clip_limit=0.01, nbins=256):
    x = rgb_uint8.astype(np.float32) / 255.0
    out = np.zeros_like(x)
    for c in range(3):
        out[..., c] = exposure.equalize_adapthist(x[..., c], clip_limit=clip_limit, nbins=nbins)
    return (np.clip(out,0,1) * 255).astype(np.uint8)

def apply_sobel(rgb_uint8):
    x = rgb_uint8.astype(np.float32) / 255.0
    gray = color.rgb2gray(x)
    edges = filters.sobel(gray)
    edges = exposure.rescale_intensity(edges, in_range='image', out_range=(0,255)).astype(np.uint8)
    return np.stack([edges, edges, edges], axis=-1)

def colorize_heatmap(hm_2d, alpha=0.35):
    cmap = plt.get_cmap('jet')
    rgba = cmap(hm_2d); rgba[...,3] = alpha
    return (rgba * 255).astype(np.uint8)

def overlay_heatmap(rgb_uint8, heat_rgba):
    base = Image.fromarray(rgb_uint8).convert('RGBA')
    heat = Image.fromarray(heat_rgba, mode='RGBA')
    base.alpha_composite(heat)
    return np.array(base.convert('RGB'))

# ---- Grad-CAM core -------------------------------------------------------------
class GradCamHook:
    def __init__(self, layer):
        self.fmap = None; self.grad = None
        self.hf = layer.register_forward_hook(self.fwd); self.hb = layer.register_full_backward_hook(self.bwd)
    def fwd(self, m, i, o): self.fmap = o.detach()
    def bwd(self, m, gi, go): self.grad = go[0].detach()
    def remove(self): self.hf.remove(); self.hb.remove()

def find_last_conv(m):
    last = None
    for _, mod in m.named_modules():
        if isinstance(mod, nn.Conv2d): last = mod
    if last is None: raise RuntimeError("No Conv2d found.")
    return last

def gradcam_maps(model, imgs, target_ids, target_layer):
    imgs = imgs.clone().detach().requires_grad_(True)
    logits = model(imgs)
    sel = logits.gather(1, target_ids.view(-1,1)).squeeze(1)
    model.zero_grad(set_to_none=True); sel.backward(torch.ones_like(sel))
    fmap, grad = target_layer._gc.fmap, target_layer._gc.grad
    w = grad.mean(dim=(2,3), keepdim=True)
    cam = (w * fmap).sum(dim=1).relu()
    cam = (cam - cam.amin((1,2),True)) / (cam.amax((1,2),True) - cam.amin((1,2),True) + 1e-6)
    cam = F.interpolate(cam[:,None], size=imgs.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)
    return cam  # (B,H,W) in [0,1]

# ---- driver --------------------------------------------------------------------
def generate_gradcam_everything(model, data_loader, device, class_names, out_dir: Path, explain='pred'):
    out_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    target_layer = find_last_conv(model.features)
    target_layer._gc = GradCamHook(target_layer)

    saved = 0
    for b, (imgs, labels) in enumerate(data_loader):
        imgs = imgs.to(device); labels = labels.to(device)
        with torch.no_grad():
            preds = model(imgs).argmax(1)
        target_ids = preds if explain == 'pred' else torch.full_like(preds, int(explain))

        cams = gradcam_maps(model, imgs, target_ids, target_layer).cpu().numpy()
        raw = denorm(imgs)                                   # (B,H,W,3) uint8
        clahe = np.stack([apply_clahe(raw[i]) for i in range(raw.shape[0])])
        sobel = np.stack([apply_sobel(raw[i]) for i in range(raw.shape[0])])

        B = imgs.size(0)
        for i in range(B):
            gt, pr, tg = int(labels[i]), int(preds[i]), int(target_ids[i])
            stem = f"idx{saved:06d}_gt-{class_names[gt]}_pred-{class_names[pr]}_tgt-{class_names[tg]}"
            Image.fromarray(raw[i]).save(out_dir / f"{stem}_image.png")
            Image.fromarray(clahe[i]).save(out_dir / f"{stem}_clahe.png")
            Image.fromarray(sobel[i]).save(out_dir / f"{stem}_sobel.png")
            heat = colorize_heatmap(cams[i])
            Image.fromarray(overlay_heatmap(clahe[i], heat)).save(out_dir / f"{stem}_gradcam.png")
            saved += 1

        if (b+1) % 20 == 0: print(f"[Explain] {saved} images done...")

    target_layer._gc.remove()
    print(f"Saved {saved} quartets to: {out_dir}")

# ---- build the “all images” loader --------------------------------------------
CSV_GLOB = "fold*_*.csv"
csvs = sorted(BASE_DIR.glob(CSV_GLOB))
dfs = [pd.read_csv(p)[["filepath","label"]] for p in csvs]
all_df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["filepath"]).reset_index(drop=True)
all_csv = BASE_DIR / "all_images_for_explain.csv"; all_df.to_csv(all_csv, index=False)
all_ds = CSVImageDataset(all_csv, transform=val_transforms)  # use your dataset/val_transforms
all_loader = DataLoader(all_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# ---- rebuild model & run -------------------------------------------------------
model = models.mobilenet_v3_large(pretrained=True)
in_feats = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_feats, len(class_map))
# model.load_state_dict(torch.load(BASE_DIR / "best_mobilenet_overall.pth", map_location=device))
model = model.to(device).eval()
class_names = [cls for cls, idx in sorted(class_map.items(), key=lambda x: x[1])]

generate_gradcam_everything(model, all_loader, device, class_names, EXPLAIN_DIR)


C:\Users\oliwa\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\oliwa\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[Explain] 640 images done...
[Explain] 1278 images done...
Saved 1278 quartets to: C:\Users\oliwa\Documents\capstone\images_for_modeling\combined_deforestation_drivers\explanations
